# 02 CodeT5-small, protokol P: FULL, QONLY, kode diacak, resplit
Satu skrip untuk semua kondisi CodeT5-small. Varian (argumen persis seperti run yang sudah ada):

| Varian | Kondisi | Seleksi checkpoint | Folder run |
|---|---|---|---|
| full_v2 / qonly_v1 | FULL / QONLY | dev BLEU | runs/codet5_full_v2, runs/codet5_qonly_v1 |
| full_bertsel / qonly_bertsel | FULL / QONLY | dev BERTScore F1 | runs/codet5_full_bertsel, runs/codet5_qonly_bertsel |
| shuffled_bertsel | dilatih dengan kode diacak | dev BERTScore F1 | runs/codet5_shuffled_bertsel |
| full_resplit_bertsel | FULL, data digabung lalu dibagi ulang (seed 42) | dev BERTScore F1 | runs/codet5_full_resplit_bertsel |

Ganti RUN, restart kernel, lalu jalankan dari atas. Satu run penuh memakan beberapa jam; untuk itu lebih aman memakai sel bg.sh.

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## codet5_p.py

In [ ]:
# Default 'smoke' (beberapa langkah saja) supaya Run All tidak langsung memulai training berjam-jam.
VARIANTS = {
    'smoke': ['--input', 'full', '--run-tag', 'codet5_full_bertsel', '--pair-tag', 'codet5_qonly_v1', '--select', 'bertscore', '--profile', 'smoke'],
    'full_v2': ['--input', 'full', '--run-tag', 'codet5_full_v2', '--pair-tag', 'codet5_qonly_v1'],
    'qonly_v1': ['--input', 'qonly', '--run-tag', 'codet5_qonly_v1', '--pair-tag', 'codet5_full_v2'],
    'full_bertsel': ['--input', 'full', '--run-tag', 'codet5_full_bertsel', '--pair-tag', 'codet5_qonly_v1', '--select', 'bertscore'],
    'qonly_bertsel': ['--input', 'qonly', '--run-tag', 'codet5_qonly_bertsel', '--pair-tag', 'codet5_full_bertsel', '--select', 'bertscore'],
    'shuffled_bertsel': ['--input', 'shuffled', '--run-tag', 'codet5_shuffled_bertsel', '--pair-tag', 'codet5_full_bertsel', '--select', 'bertscore'],
    'full_resplit_bertsel': ['--input', 'full', '--run-tag', 'codet5_full_resplit_bertsel', '--pair-tag', 'codet5_full_bertsel', '--select', 'bertscore', '--resplit', '42'],
}
RUN = 'smoke'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['codet5_p.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/codet5_p.py

```text
E1: CodeT5-small FULL vs QONLY under Protocol P (single seed, W&B).

Protocol P: AdamW lr 5e-5, effective batch 64, 20 epochs, no early stopping,
best epoch by dev BLEU (beam 10 on all 7,000 dev), test beam 10, source 400,
answer 30, question before code. FULL and QONLY differ only in the input.
A dev-F1-selected checkpoint is kept as a secondary (analysis D only).
```

In [ ]:
import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup

import kc_common as kc

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--input", choices=("full", "qonly", "shuffled"), required=True)
parser.add_argument("--run-tag", required=True)
parser.add_argument("--pair-tag", required=True, help="run tag of the counterpart condition")
parser.add_argument("--profile", choices=("smoke", "full"), default="full")
parser.add_argument("--micro-batch", type=int, default=64)
parser.add_argument("--eval-batch", type=int, default=96)
parser.add_argument("--select", choices=("bleu", "bertscore"), default="bleu",
                    help="dev metric that picks the reported checkpoint; declared before training")
parser.add_argument("--resplit", type=int, default=None, help="pool train+dev+test, reshuffle with this seed into 56085/7000/7000")
args = parser.parse_args(ARGS)
SELECT_KEY = "dev_BERTScore_F1" if args.select == "bertscore" else "dev_BLEU"

SEED = 42
MODEL_ID = "Salesforce/codet5-small"
EFFECTIVE_BATCH = 64
assert EFFECTIVE_BATCH % args.micro_batch == 0
smoke = args.profile == "smoke"
CONFIG = dict(
    experiment="E1", protocol="P", model=MODEL_ID, condition=args.input.upper(), input_mode=args.input,
    pair_run_tag=args.pair_tag, template_full="question: {question}\\ncode: {code}",
    template_used=("question: {question}\\ncode: {code}" if args.input == "full" else "question: {question}\\ncode: "),
    max_source=400, max_target=30, optimizer="AdamW", lr=5e-5, adam_eps=1e-8, weight_decay=0.0,
    warmup_fraction=0.1, scheduler="linear", max_grad_norm=1.0, effective_batch=EFFECTIVE_BATCH,
    micro_batch=args.micro_batch, grad_accumulation=EFFECTIVE_BATCH // args.micro_batch,
    epochs=2 if smoke else 20, early_stopping=False, selection=SELECT_KEY, secondary_selection="dev_F1",
    dev_size=64 if smoke else 7000, dev_decoding="beam10", test_beam=10, generation_max_new_tokens=30,
    decode="skip_special_tokens=True, clean_up_tokenization_spaces=False", seed=SEED,
    precision="bf16 autocast (fp32 master weights)", profile=args.profile,
    deviations=["single shared A100: FULL and QONLY train concurrently on one GPU",
                "bf16 autocast instead of the old CodeT5 run's fp32 (identical for both conditions)",
                "source 400 / answer 30 per Protocol P; the old CodeT5 FULL used 512 / 64",
                "max_grad_norm 1.0 (not specified by Protocol P; same for both conditions)",
                "W&B credentials come from the account netrc, not WANDB_API_KEY"],
)

In [ ]:
RUN_TAG = f"smoke_{args.run_tag}" if smoke else args.run_tag
if smoke:
    import os
    os.environ.setdefault("WANDB_MODE", "offline")  # smoke runs never land in the project
out = kc.EXP / "runs" / RUN_TAG
out.mkdir(parents=True, exist_ok=True)

In [ ]:
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [ ]:
seed_all(SEED)
device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
rows = {split: kc.load_split(split) for split in ("train", "dev", "test")}
if args.resplit is not None:
    pool = [dict(r, orig=f"{s}:{r['id']}") for s in ("train", "dev", "test") for r in rows[s]]
    assert len(pool) == 70085
    for idx, r in enumerate(pool):
        r["id"] = idx
    random.Random(args.resplit).shuffle(pool)
    sizes = ("train", 56085), ("dev", 7000), ("test", 7000)
    start = 0
    for name, n in sizes:
        rows[name] = pool[start:start + n]
        start += n
    kc.write_json(kc.EXP / "maps" / f"resplit_seed{args.resplit}.json", {s: [r["orig"] for r in rows[s]] for s, _ in sizes})
if smoke:
    rows = {"train": rows["train"][:256], "dev": rows["dev"][:64], "test": rows["test"][:64]}

In [ ]:
def shuffle_code(split_rows):
    """Derangement of the code column within one split, same rule as make_maps.py (E2), seed 42."""
    codes = [r["code"] for r in split_rows]
    n = len(codes)
    rng = random.Random(SEED)
    perm = list(range(n))
    for _ in range(1000):
        rng.shuffle(perm)
        if all(perm[i] != i and codes[perm[i]] != codes[i] for i in range(n)):
            break
    else:
        raise RuntimeError("no valid derangement found")
    return [{**r, "code": codes[perm[i]]} for i, r in enumerate(split_rows)], perm

In [ ]:
CODE_IDS = {}
if args.input == "shuffled":
    for split in rows:
        ids = [r["id"] for r in rows[split]]
        rows[split], perm = shuffle_code(rows[split])
        CODE_IDS[split] = [ids[j] for j in perm]

In [ ]:
def code_ids_for(split):
    if args.input == "shuffled":
        return CODE_IDS[split]
    return [r["id"] if args.input == "full" else None for r in rows[split]]

In [ ]:
TEMPLATE = "qonly" if args.input == "qonly" else "full"  # shuffled keeps the FULL template, donor code

In [ ]:
def encode(split_rows):
    sources = [kc.codet5_source(r["question"], r["code"], TEMPLATE) for r in split_rows]
    enc = tokenizer(sources, max_length=CONFIG["max_source"], truncation=True)
    lab = tokenizer(text_target=[r["answer"] for r in split_rows], max_length=CONFIG["max_target"], truncation=True)
    full_len = [len(x) for x in tokenizer([kc.codet5_source(r["question"], r["code"], "full") for r in split_rows])["input_ids"]]
    prefix_len = [len(x) for x in tokenizer([kc.codet5_source(r["question"], r["code"], "qonly") for r in split_rows])["input_ids"]]
    if max(prefix_len) >= CONFIG["max_source"]:
        raise ValueError("a question leaves no code budget under the 400-token source limit")
    stats = dict(n=len(split_rows), full_source_truncated=sum(n > CONFIG["max_source"] for n in full_len),
                 target_truncated=sum(len(x) >= CONFIG["max_target"] for x in lab["input_ids"]))
    return enc["input_ids"], lab["input_ids"], stats

In [ ]:
encoded = {split: encode(r) for split, r in rows.items()}
CONFIG["length_stats"] = {split: e[2] for split, e in encoded.items()}
CONFIG["data_sha256"] = kc.data_hashes()
CONFIG["metric_provenance"] = kc.codeqa_metrics.metric_provenance(kc.REPO)
CONFIG["torch"] = torch.__version__

In [ ]:
import transformers  # noqa: E402

In [ ]:
CONFIG["transformers"] = transformers.__version__

In [ ]:
def assert_no_code(batch_ids, batch_rows):
    """QONLY: the encoder input must equal a fresh tokenisation of the question-only template."""
    for ids, row in zip(batch_ids, batch_rows):
        expected = tokenizer(kc.codet5_source(row["question"], row["code"], "qonly"),
                             max_length=CONFIG["max_source"], truncation=True)["input_ids"]
        if list(ids) != expected:
            raise AssertionError(f"QONLY encoder input differs from question-only template for id {row['id']}")
        code_ids = tokenizer(row["code"], add_special_tokens=False)["input_ids"][:8]
        if len(code_ids) >= 4 and any(list(ids[i:i + len(code_ids)]) == code_ids for i in range(len(ids))):
            raise AssertionError(f"code tokens reached the encoder for id {row['id']}")

In [ ]:
def batches(split, order, size, with_labels):
    src, lab, _ = encoded[split]
    pad = tokenizer.pad_token_id
    for start in range(0, len(order), size):
        idx = order[start:start + size]
        width = max(len(src[i]) for i in idx)
        ids = torch.full((len(idx), width), pad, dtype=torch.long)
        mask = torch.zeros((len(idx), width), dtype=torch.long)
        for j, i in enumerate(idx):
            ids[j, :len(src[i])] = torch.tensor(src[i]); mask[j, :len(src[i])] = 1
        batch = {"input_ids": ids, "attention_mask": mask}
        if with_labels:
            lw = max(len(lab[i]) for i in idx)
            labels = torch.full((len(idx), lw), -100, dtype=torch.long)
            for j, i in enumerate(idx):
                labels[j, :len(lab[i])] = torch.tensor(lab[i])
            batch["labels"] = labels
        yield idx, batch

In [ ]:
@torch.no_grad()
def generate(model, split):
    model.eval()
    src = encoded[split][0]
    order = sorted(range(len(src)), key=lambda i: -len(src[i]))  # length-sorted for speed; order restored below
    predictions = [None] * len(src)
    for idx, batch in batches(split, order, args.eval_batch, with_labels=False):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            generated = model.generate(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device),
                                       num_beams=10, max_new_tokens=CONFIG["generation_max_new_tokens"], do_sample=False)
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        for i, text in zip(idx, texts):
            predictions[i] = text.strip()
    model.train()
    return predictions

In [ ]:
def score(split, predictions):
    return kc.evaluate(predictions, [r["answer"] for r in rows[split]])

In [ ]:
# --- resume only with identical config ---
config_path = out / "config.json"
if config_path.exists():
    if json.loads(config_path.read_text()) != json.loads(json.dumps(CONFIG)):
        raise SystemExit(f"{out} exists with a different config; use a new run tag instead of changing settings")
else:
    kc.write_json(config_path, CONFIG)

model = T5ForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
CONFIG_PARAMS = sum(p.numel() for p in model.parameters())
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], eps=CONFIG["adam_eps"], weight_decay=CONFIG["weight_decay"])
steps_per_epoch = -(-len(rows["train"]) // EFFECTIVE_BATCH)
total_steps = steps_per_epoch * CONFIG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(CONFIG["warmup_fraction"] * total_steps), total_steps)

history, start_epoch = [], 1
best = {"dev_BLEU": -1.0, "dev_BLEU_epoch": None, "dev_F1": -1.0, "dev_F1_epoch": None,
        "dev_BERTScore_F1": -1.0, "dev_BERTScore_F1_epoch": None}
last_path = out / "last.pt"
if last_path.exists():
    state = torch.load(last_path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"]); scheduler.load_state_dict(state["scheduler"])
    torch.set_rng_state(state["rng_cpu"]); torch.cuda.set_rng_state(state["rng_cuda"])
    random.setstate(state["rng_py"]); np.random.set_state(state["rng_np"])
    history, best, start_epoch = state["history"], state["best"], state["epoch"] + 1
    print(f"Resuming {RUN_TAG} from epoch {state['epoch']} with identical config", flush=True)

run = kc.wandb_init(out, RUN_TAG, "E1", ["codet5", CONFIG["condition"], "protocolP", "seed42"],
                    {**CONFIG, "params": CONFIG_PARAMS}, "train")
accum = CONFIG["grad_accumulation"]
model.train()

In [ ]:
for epoch in range(start_epoch, CONFIG["epochs"] + 1):
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    order = torch.randperm(len(rows["train"]), generator=torch.Generator().manual_seed(SEED * 1000 + epoch)).tolist()
    check_batches = {0, 1, len(order) // args.micro_batch // 2}
    losses = []
    for b, (idx, batch) in enumerate(batches("train", order, args.micro_batch, with_labels=True)):
        if args.input == "qonly" and b in check_batches:
            assert_no_code([encoded["train"][0][i] for i in idx], [rows["train"][i] for i in idx])
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss
        (loss / accum).backward()
        losses.append(loss.item())
        if (b + 1) % accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    train_seconds = time.time() - t0
    if args.input == "qonly":
        assert_no_code(encoded["dev"][0][:32], rows["dev"][:32])
    t1 = time.time()
    dev_predictions = generate(model, "dev")
    dev_metrics, dev_per = score("dev", dev_predictions)
    if args.select == "bertscore":  # dev only; test is never looked at for selection
        import bert_score  # shortcut: roberta-large default model, no baseline rescaling
        _, _, bf = bert_score.score(dev_predictions, [r["answer"] for r in rows["dev"]], lang="en",
                                    batch_size=128, verbose=False, device="cuda")
        dev_metrics["BERTScore_F1"] = 100 * bf.mean().item()
    dev_seconds = time.time() - t1
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    entry = dict(epoch=epoch, train_loss=float(np.mean(losses)), train_seconds=train_seconds, dev_seconds=dev_seconds,
                 epoch_seconds=train_seconds + dev_seconds, peak_gpu_mem_gb=peak_gb, lr=scheduler.get_last_lr()[0],
                 **{f"dev_{k}": v for k, v in dev_metrics.items() if isinstance(v, (int, float))})
    history.append(entry)
    (out / "dev_predictions").mkdir(exist_ok=True)
    kc.write_predictions(out / "dev_predictions" / f"epoch_{epoch:02d}.jsonl", rows["dev"], dev_predictions, dev_per,
                         CONFIG["condition"], code_ids_for("dev"))
    for key, folder in (("BLEU", "best_bleu"), ("F1", "best_f1"), ("BERTScore_F1", "best_bertscore")):
        if key not in dev_metrics:
            continue
        if dev_metrics[key] > best[f"dev_{key}"]:
            best[f"dev_{key}"], best[f"dev_{key}_epoch"] = dev_metrics[key], epoch
            model.save_pretrained(out / folder); tokenizer.save_pretrained(out / folder)
    run.log({"epoch": epoch, "train/loss": entry["train_loss"], "train/lr": entry["lr"],
             "time/epoch_seconds": entry["epoch_seconds"], "time/train_seconds": train_seconds, "time/dev_seconds": dev_seconds,
             "gpu/peak_mem_gb": peak_gb, **{f"dev/{k}": v for k, v in dev_metrics.items() if isinstance(v, (int, float))}})
    kc.write_json(out / "history.json", history)
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state(), "rng_py": random.getstate(),
                "rng_np": np.random.get_state(), "history": history, "best": best, "epoch": epoch}, out / "last.pt.tmp")
    (out / "last.pt.tmp").replace(last_path)
    print(f"epoch {epoch}: loss {entry['train_loss']:.4f} dev BLEU {dev_metrics['BLEU']:.2f} F1 {dev_metrics['F1']:.2f} "
          f"EM {dev_metrics['EM']:.2f} ({entry['epoch_seconds']:.0f}s, peak {peak_gb:.1f} GB)", flush=True)

In [ ]:
kc.write_json(out / "status.json", {"training_complete": True, "epochs": CONFIG["epochs"], **best})

In [ ]:
def test_with(folder, tag):
    m = T5ForConditionalGeneration.from_pretrained(out / folder).to(device)
    predictions = generate(m, "test")
    metrics, per = score("test", predictions)
    del m
    torch.cuda.empty_cache()
    return predictions, metrics, per

In [ ]:
test_rows = rows["test"]
code_ids = code_ids_for("test")
primary = "best_bertscore" if args.select == "bertscore" else "best_bleu"
predictions, metrics, per = test_with(primary, "primary")
results = {"run_tag": RUN_TAG, "condition": CONFIG["condition"], "selection": SELECT_KEY,
           "selected_epoch": best[f"{SELECT_KEY}_epoch"], f"best_{SELECT_KEY}": best[SELECT_KEY], "test_beam": 10,
           "evaluation_scope": "smoke" if smoke else "full_test", "test": metrics}
kc.write_json(out / "results.json", results)
kc.write_predictions(out / "test_predictions.jsonl", test_rows, predictions, per, CONFIG["condition"], code_ids)
# Secondary (analysis D only): dev-F1-selected checkpoint, never used to choose anything.
p2, m2, per2 = test_with("best_f1", "devF1")
kc.write_json(out / "results_devF1_selected.json", {"run_tag": RUN_TAG, "selection": "dev_F1 (secondary)",
              "selected_epoch": best["dev_F1_epoch"], "best_dev_F1": best["dev_F1"], "test": m2})
kc.write_predictions(out / "test_predictions_devF1_selected.jsonl", test_rows, p2, per2, CONFIG["condition"], code_ids)
run.summary.update({f"test_devF1sel/{k}": v for k, v in m2.items() if isinstance(v, (int, float)) and not isinstance(v, bool)})
records = [json.loads(line) for line in (out / "test_predictions.jsonl").read_text(encoding="utf-8").splitlines()]
kc.wandb_finalize(run, out, RUN_TAG, metrics, records)
run.finish()
print("DONE", json.dumps({k: metrics[k] for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}), flush=True)

In [ ]:
# Opsional untuk run panjang: jalankan varian terpilih di tmux lewat bg.sh (log di logs/<RUN>.log),
# supaya training tidak mati kalau browser/kernel putus. Hapus tanda # untuk memakai.
# import subprocess
# subprocess.run([str(EXP / "bg.sh"), RUN, sys.executable, 'codet5_p.py', *ARGS], check=True)

### Ringkasan hasil test (dibaca dari runs/<tag>/results.json)

In [ ]:
TAGS = ['codet5_full_bertsel', 'codet5_full_v2', 'codet5_qonly_v1', 'codet5_full_bertsel', 'codet5_qonly_bertsel', 'codet5_shuffled_bertsel', 'codet5_full_resplit_bertsel']
for tag in TAGS:
    for cand in (tag, "smoke_" + tag):
        f = EXP / "runs" / cand / "results.json"
        if f.exists():
            r = json.loads(f.read_text())
            t = r.get("test", {})
            print(cand, {k: round(v, 2) for k, v in t.items() if isinstance(v, (int, float))})
            break
    else:
        print(tag, "-> belum ada results.json")

## bertscore_codet5.py
BERTScore untuk prediksi test codet5_full_v2 dan codet5_qonly_v1 (run seleksi BLEU).

In [ ]:
ARGS = []
sys.argv = ['bertscore_codet5.py']

**Sumber:** code/bertscore_codet5.py

In [ ]:
import json, bert_score

In [ ]:
E = "/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp"
for tag in ["codet5_full_v2", "codet5_qonly_v1"]:
    rows = [json.loads(l) for l in open(f"{E}/runs/{tag}/test_predictions.jsonl", encoding="utf-8")]
    assert len(rows) == 7000
    P, R, F = bert_score.score([r["prediction"] for r in rows], [r["reference"] for r in rows],
                               lang="en", batch_size=128, verbose=False, device="cuda")
    with open(f"{E}/runs/{tag}/bertscore.jsonl", "w", encoding="utf-8") as f:
        for r, p, rc, fs in zip(rows, P.tolist(), R.tolist(), F.tolist()):
            f.write(json.dumps({"id": r["id"], "BERT_P": p, "BERT_R": rc, "BERT_F1": fs}) + "\n")
    summ = {"run_tag": tag, "n": len(rows), "model": "bert_score default English (roberta-large), no baseline rescaling",
            "BERTScore_P": 100 * P.mean().item(), "BERTScore_R": 100 * R.mean().item(), "BERTScore_F1": 100 * F.mean().item()}
    json.dump(summ, open(f"{E}/runs/{tag}/results_bertscore.json", "w"), indent=2)
    print("DONE", json.dumps(summ), flush=True)